In [6]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
import torch
from torch import nn
import pandas as pd
from huggingface_hub import login
from datasets import load_dataset

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s | %(levelname)s | %(message)s",
)

torch.manual_seed(123)

# src_path = Path.cwd().parent / "src"
# if src_path.exists() and str(src_path) not in sys.path:
#     sys.path.insert(0, str(src_path))

# print(f"Added to sys.path: {src_path}")
load_dotenv()  # reads .env file from the current directory

PATH_DATA = Path.cwd().parent.parent / ".data"
PATH_GPT2_124M_WEIGHTS = PATH_DATA / "model_weights" / "gpt2" / "124M" / "parameters.pickle.gz"
# DATASET = "openchat/ultrachat-sharegpt"
DATASET="~/Downloads/alpaca_gpt4_data.json"

login(os.getenv("HF_TOKEN"))

httpx | INFO | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
huggingface_hub._login | WARNING | Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    formatted_input = instruction_text + input_text + f"\n\n### Response:\n{entry['output']}"
    return {"texts": formatted_input}

In [8]:
from datasets import Dataset as HFDataset
from typing import Tuple

class InstructionsDataset(HFDataset):
    """Dataset class for tokenized text with automatic padding and truncation.

    This class extends Hugging Face Dataset to handle text tokenization,
    padding, and truncation for machine learning models.
    """

    def custom_collate_fn(
            self, 
        input_ids,
        pad_token_id=50256,
        ignore_index=-100,
        allowed_max_length=None
    ) -> Tuple[list[list[int]], list[list[int]]]:
        batch_max_length = max(len(item)+1 for item in input_ids)
        inputs_lst, targets_lst = [], []

        for item in input_ids:
            new_item = item.copy()
            new_item += [pad_token_id]

            padded = (                               #1
                new_item + [pad_token_id] *          #1
                (batch_max_length - len(new_item))   #1 Pads sequences to max_length
            )
            inputs = torch.tensor(padded[:-1])      #2 Truncates the last token for inputs
            targets = torch.tensor(padded[1:])     #3 Shifts +1 to the right for targets

            mask = targets == pad_token_id              #4
            indices = torch.nonzero(mask).squeeze()     #4
            if indices.numel() > 1:                     #4
                targets[indices[1:]] = ignore_index     #4 Replaces all but the first padding tokens in targets by ignore_index

            if allowed_max_length is not None:
                inputs = inputs[:allowed_max_length]       #5
                targets = targets[:allowed_max_length]     #5 Optionally truncates to the maximum sequence length

            inputs_lst.append(inputs.tolist())
            targets_lst.append(targets.tolist())

        return inputs_lst, targets_lst

    def __init__(
        self,
        tokenizer,
        texts: list[str],
        max_length=None,
        pad_token_id=50256,
    ) -> None:
        """Initialize InstructionsDataset with tokenized text.
        Args:
            tokenizer: Tokenizer instance with encode method.
            texts: List of text strings to tokenize.
            max_length: Maximum sequence length. If None, uses longest encoded text.
            pad_token_id: Token ID used for padding sequences (default: 50256).
        """
        self.texts = texts

        # 1 Pretokenizes texts
        self.encoded_texts = [tokenizer.encode(text) for text in self.texts]
        inputs, targets = self.custom_collate_fn(input_ids=self.encoded_texts, pad_token_id=pad_token_id, allowed_max_length=max_length)

        hf_dataset = HFDataset.from_dict(
            {
                "inputs": inputs, "targets": targets
            }
        )

        super().__init__(
            arrow_table=hf_dataset._data,  # noqa: SLF001
            info=hf_dataset.info,
            split=hf_dataset.split,
            indices_table=hf_dataset._indices,  # noqa: SLF001
            fingerprint=hf_dataset._fingerprint,  # noqa: SLF001
        )

    def _longest_encoded_length(self) -> int:
        """Return the length of the longest encoded text."""
        max_length = 0
        for encoded_text in self.encoded_texts:
            max_length = max(max_length, len(encoded_text))
        return max_length

In [9]:
source_ds = (load_dataset("json", data_files="../../.data/alpaca_gpt4_data.json")["train"]).map(format_input, remove_columns=["instruction", "input", "output"])
ds = source_ds.train_test_split(test_size=0.2)
ds_train = ds["train"]
ds_test = ds["test"]
ds_val = ds_test.train_test_split(test_size=0.5)
ds_test = ds_val["train"]
ds_val = ds_val["test"]

httpx | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"


In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
ds_train = InstructionsDataset(tokenizer=tokenizer, texts=list(ds_train["texts"]))
ds_test = InstructionsDataset(tokenizer=tokenizer, texts=list(ds_test["texts"]))
ds_val = InstructionsDataset(tokenizer=tokenizer, texts=list(ds_val["texts"]))

In [ ]:
from tgedr_lm.configuration import ClassifierBaseConfiguration, TrainingArgs
from tgedr_lm.commons.utils_io import load_pickle_compressed
from tgedr_lm.classifier.gpt2.model import GPT2Classifier

weights = load_pickle_compressed(PATH_GPT2_124M_WEIGHTS)

In [17]:
model = GPT2Classifier(ClassifierBaseConfiguration(n_classes=3))
model.pretrain(weights)

In [13]:
print(len(ds_val['inputs']))
ds_val['inputs'][0][0:3]

5201


[21106, 318, 281]

In [14]:
print(len(ds_val['targets']))
ds_val['targets'][0][0:3]

5201


[318, 281, 12064]

In [ ]:
ds = source_ds.train_test_split(test_size=0.2)


In [ ]:
data = {
        "train": {"input": list(ds_train["input"]), "output": list(ds_train["output"])},
        "test": {"input": list(ds_test["input"]), "output": list(ds_test["output"])},
        "validation": {"input": list(ds_val["input"]), "output": list(ds_val["output"])}
}